In [21]:

import os
import gc
import json
import random
import logging
from typing import List, Dict, Tuple


import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import DataLoader


import torchvision
from torchvision import datasets, transforms, models


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn

from sklearn.metrics import confusion_matrix

# =========================
# NLP / Hugging Face
# =========================
import sacrebleu
from datasets import load_dataset, load_from_disk

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)


from peft import LoraConfig, TaskType, get_peft_model, PeftModel


from tqdm import tqdm

import os, gc, re, json, random, logging
from collections import defaultdict
from pathlib import Path
 
import numpy as np
import sacrebleu
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import nltk
from rouge_score import rouge_scorer
from datasets import load_dataset, load_from_disk, concatenate_datasets, Dataset as HFDataset, DatasetDict, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, AutoModel,
    DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
import torch.nn as nn
 
nltk.download("wordnet", quiet=True)
nltk.download("punkt",   quiet=True)
 
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
 




def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

DEVICE = get_device()
print(f"Using device: {DEVICE}")

Using device: mps


#### test



In [ ]:
urdu_eng= load_dataset("HaiderSultanArc/MT-Urdu-English")
print(urdu_eng.items())
#shuffle and select 1000 samples for training and 100 for testing
urdu_eng = urdu_eng.shuffle(seed=42)
urdu_eng_2000 = urdu_eng["train"].select(range(2000))
del urdu_eng
gc.collect()


In [ ]:
legal_uqa = load_dataset("nlp-anonymous-researcher/LEGAL-UQA")
print(legal_uqa.items())

In [ ]:
idioms= load_dataset("Ehtisham1328/urdu-idioms-with-english-translation")
print(idioms.items())

In [ ]:
alignemnt_data = load_dataset("Lots-of-LoRAs/task1057_pib_translation_english_urdu")
print(alignemnt_data.items())

# loading and merging datasets


In [11]:

SEED = 42
random.seed(SEED)

OUTPUT_DIR = Path("en_ur_output")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "split").mkdir(exist_ok=True)

In [12]:
def clean_dataset(dataset: HFDataset) -> HFDataset:
    def is_valid(row):
        en, ur = row["en"].strip(), row["ur"].strip()
        if re.match(r"^\d+[\.\d]*\s", en) or re.match(r"^\d+[\.\d]*\s", ur):
            return False
        en_len, ur_len = len(en.split()), len(ur.split())
        if ur_len == 0 or not (0.3 <= en_len / ur_len <= 3.0):
            return False
        return True
    cleaned = dataset.filter(is_valid, desc="Cleaning")
    print(f"  Before: {len(dataset):,}  →  After: {len(cleaned):,}  "
          f"(removed {len(dataset)-len(cleaned):,}, "
          f"{(len(dataset)-len(cleaned))/len(dataset)*100:.1f}%)")
    return cleaned
 
 


In [13]:
def clean(text: str) -> str:
    """Strip excess whitespace and newlines."""
    if not isinstance(text, str):
        return ""
    return " ".join(text.split()).strip()


In [14]:
def dedupe(pairs: list[dict]) -> list[dict]:
    seen = set()
    out = []
    for p in pairs:
        key = (p["en"], p["ur"])
        if key not in seen and p["en"] and p["ur"]:
            seen.add(key)
            out.append(p)
    return out


In [15]:

def sample(pairs: list[dict], n: int) -> list[dict]:
    """Random sample without replacement."""
    return random.sample(pairs, min(n, len(pairs)))


In [17]:
def extract_ds1() -> list[dict]:
    print("Loading DS1: HaiderSultanArc/MT-Urdu-English ...")
    ds = load_dataset("HaiderSultanArc/MT-Urdu-English")
    pairs = []
    for split in ds:
        for row in ds[split]:
            row: dict

            en = clean(row.get("en", ""))
            ur = clean(row.get("ur", ""))
            if en and ur:
                pairs.append({"en": en, "ur": ur, "source": "MT-Urdu-English"})
    pairs = dedupe(pairs)
    print(f"  → {len(pairs)} unique pairs")
    return pairs


In [ ]:
def extract_ds2() -> list[dict]:
  
    print("Loading DS2: LEGAL-UQA ...")
    ds = load_dataset("nlp-anonymous-researcher/LEGAL-UQA")
    pairs = []
    field_pairs = [
        ("question_eng", "question_urdu"),
        ("context_eng",  "context_urdu"),
        ("answer_eng",   "answer_urdu"),
    ]
    for split in ds:
        for row in ds[split]:
            for eng_col, ur_col in field_pairs:
                en = clean(row.get(eng_col, ""))
                ur = clean(row.get(ur_col, ""))
                if en and ur:
                    pairs.append({"en": en, "ur": ur, "source": "LEGAL-UQA"})
    pairs = dedupe(pairs)
    print(f"  → {len(pairs)} unique pairs")
    return pairs



In [ ]:
def extract_ds3() -> list[dict]:
    """
    Format: 'english_word: urdu_word'
    We parse both sides. Note: Urdu side may actually be the romanised word;
    the dataset is transliteration-focused. We keep both sides as-is and
    label them clearly so downstream filtering is easy.
    """
    print("Loading DS3: Urdu-English Transliteration ...")
    ds = load_dataset("irtizaab/Urdu-English_Transliteration")
    pairs = []
    for split in ds:
        for row in ds[split]:
            text = row.get("text", "")
            if ":" not in text:
                continue
            # split on FIRST colon only
            parts = text.split(":", 1)
            roman = clean(parts[0])   # e.g. "aab"
            eng   = clean(parts[1])   # e.g. "water"
            if roman and eng:
                # store english in 'en', romanised Urdu in 'ur'
                # (these are word-level transliteration pairs)
                pairs.append({"en": eng, "ur": roman, "source": "Transliteration"})
    pairs = dedupe(pairs)
    print(f"  → {len(pairs)} unique pairs (all will be kept)")
    return pairs



In [19]:
def extract_ds4() -> list[dict]:
    
    print("Loading DS4: task1057 PIB Translation ...")
    ds = load_dataset("Lots-of-LoRAs/task1057_pib_translation_english_urdu")
    pairs = []

    # Regex: grab text between last "Input:" and "Output:"
    pattern = re.compile(
        r"Now complete the following example\s*-\s*\nInput:\s*(.+?)\nOutput:",
        re.DOTALL,
    )

    for split in ds:
        for row in ds[split]:
            inp = row.get("input", "")
            out = row.get("output", [])

            m = pattern.search(inp)
            if not m:
                continue
            en = clean(m.group(1))
            ur = clean(out[0]) if isinstance(out, list) and out else clean(str(out))

            if en and ur:
                pairs.append({"en": en, "ur": ur, "source": "PIB-Translation"})

    pairs = dedupe(pairs)
    print(f"  → {len(pairs)} unique pairs")
    return pairs



# claude 

In [22]:
def main():
    ds1_pairs = extract_ds1()
    ds2_pairs = extract_ds2()
    ds3_pairs = extract_ds3()
    ds4_pairs = extract_ds4()

    # Balance DS1, DS2, DS4 to the size of the smallest
    cap = min(len(ds1_pairs), len(ds2_pairs), len(ds4_pairs))
    print(f"\nBalancing DS1 / DS2 / DS4 to {cap} pairs each")
    print(f"DS3 kept in full: {len(ds3_pairs)} pairs")

    ds1_sampled = sample(ds1_pairs, cap)
    ds2_sampled = sample(ds2_pairs, cap)
    ds4_sampled = sample(ds4_pairs, cap)

    # Free raw lists immediately after sampling
    del ds1_pairs, ds2_pairs, ds3_pairs, ds4_pairs
    gc.collect()

    combined = ds1_sampled + ds2_sampled + ds4_sampled
    del ds1_sampled, ds2_sampled, ds4_sampled
    gc.collect()

    random.shuffle(combined)
    total = len(combined)
    print(f"\nTotal combined pairs: {total}")
    print(f"  MT-Urdu-English : {cap}")
    print(f"  LEGAL-UQA       : {cap}")
    print(f"  PIB-Translation : {cap}")

    # ── save flat files ───────────────────────────────────────────────────
    df = pd.DataFrame(combined)
    csv_path = OUTPUT_DIR / "en_ur_parallel.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    del df
    print(f"\nSaved CSV → {csv_path}")

    jsonl_path = OUTPUT_DIR / "en_ur_parallel.jsonl"
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for row in combined:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Saved JSONL → {jsonl_path}")

    # ── train / validation / test split (90 / 5 / 5) ─────────────────────
    n_val   = max(1, int(total * 0.05))
    n_test  = max(1, int(total * 0.05))
    n_train = total - n_val - n_test

    train_data = combined[:n_train]
    val_data   = combined[n_train : n_train + n_val]
    test_data  = combined[n_train + n_val :]
    del combined
    gc.collect()

    for name, data in [("train", train_data), ("validation", val_data), ("test", test_data)]:
        path = OUTPUT_DIR / "split" / f"{name}.jsonl"
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            for row in data:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(f"  {name:10s}: {len(data):>6,} pairs → {path}")

    # ── HuggingFace DatasetDict ───────────────────────────────────────────
    hf_dataset = DatasetDict({
        "train":      Dataset.from_list(train_data),
        "validation": Dataset.from_list(val_data),
        "test":       Dataset.from_list(test_data),
    })
    del train_data, val_data, test_data
    gc.collect()

    hf_path = OUTPUT_DIR / "hf_dataset"
    hf_dataset.save_to_disk(str(hf_path))
    del hf_dataset
    gc.collect()
    print(f"\nHuggingFace DatasetDict saved → {hf_path}")
    print("\nDone! ✓")


if __name__ == "__main__":
    main()

Loading DS1: HaiderSultanArc/MT-Urdu-English ...


INFO: HTTP Request: HEAD https://huggingface.co/datasets/HaiderSultanArc/MT-Urdu-English/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/HaiderSultanArc/MT-Urdu-English/fed60ae5090985578e4d71c02a29cd51232e40f9/README.md "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/datasets/HaiderSultanArc/MT-Urdu-English/resolve/fed60ae5090985578e4d71c02a29cd51232e40f9/MT-Urdu-English.py "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/HaiderSultanArc/MT-Urdu-English/HaiderSultanArc/MT-Urdu-English.py "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET https://huggingface.co/api/datasets/HaiderSultanArc/MT-Urdu-English/revision/fed60ae5090985578e4d71c02a29cd51232e40f9 "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/datasets/HaiderSultanArc/MT-Urdu-English/resolve/fed60ae5090985578e4d71c02a29cd51232e40f9/.huggingface.y

KeyboardInterrupt: 

nllb final py

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Install dependencies
# ══════════════════════════════════════════════════════════════════════════════
# !pip install -q transformers datasets peft accelerate sacrebleu sentencepiece
# !pip install -q rouge-score nltk torchao matplotlib

# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Verify GPU
# ══════════════════════════════════════════════════════════════════════════════
import torch
assert torch.cuda.is_available(), "❌ No GPU — Runtime → Change runtime type → T4 GPU"
print(f"✅ GPU  : {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Imports & global config
# ══════════════════════════════════════════════════════════════════════════════
import os, gc, re, json, random, logging
from collections import defaultdict
from pathlib import Path

import numpy as np
import sacrebleu
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import nltk
from rouge_score import rouge_scorer
from datasets import load_dataset, load_from_disk, concatenate_datasets, Dataset as HFDataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, AutoModel,
    DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
import torch.nn as nn

nltk.download("wordnet", quiet=True)
nltk.download("punkt",   quiet=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ── Constants ─────────────────────────────────────────────────────────────────
NLLB_ID   = "facebook/nllb-200-distilled-600M"
XLMR_ID   = "xlm-roberta-base"      # 768-dim encoder
SRC_LANG  = "eng_Latn"
TGT_LANG  = "urd_Arab"
MAX_LEN   = 128
SEED      = 42
N_SAMPLES = 5
N_ERROR   = 300    # samples for error analysis per experiment

# Output dirs — one per experiment
DIR_EXP1 = "exp1_unclean"
DIR_EXP2 = "exp2_clean"
DIR_EXP3 = "exp3_combined"
DIR_EXP4 = "exp4_xlmr"

LORA_CFG = dict(
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
    bias="none", task_type=TaskType.SEQ_2_SEQ_LM,
)
TRAIN_ARGS = dict(
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=3e-4,
    fp16=True,
    optim="adafactor",
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
)

random.seed(SEED)
print("✅ Config ready")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Data cleaning function
# ══════════════════════════════════════════════════════════════════════════════
def clean_dataset(dataset: HFDataset) -> HFDataset:
    """Remove noisy pairs: numbered section headers + length-ratio outliers."""
    def is_valid(row):
        en, ur = row["en"].strip(), row["ur"].strip()
        if re.match(r"^\d+[\.\d]*\s", en) or re.match(r"^\d+[\.\d]*\s", ur):
            return False
        en_len, ur_len = len(en.split()), len(ur.split())
        if ur_len == 0 or not (0.3 <= en_len / ur_len <= 3.0):
            return False
        return True
    cleaned = dataset.filter(is_valid, desc="Cleaning")
    print(f"  Before: {len(dataset):,}  →  After: {len(cleaned):,}  "
          f"(removed {len(dataset)-len(cleaned):,}, "
          f"{(len(dataset)-len(cleaned))/len(dataset)*100:.1f}%)")
    return cleaned

print("✅ clean_dataset() ready")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Load all 4 datasets & normalise to {en, ur, source}
# ══════════════════════════════════════════════════════════════════════════════
print("Loading datasets...")

# ── DS1: MT-Urdu-English ──────────────────────────────────────────────────────
_d1      = load_dataset("HaiderSultanArc/MT-Urdu-English")
ds1_full = concatenate_datasets([_d1[s] for s in _d1.keys()])
ds1_full = ds1_full.map(lambda r: {"en": r["en"], "ur": r["ur"], "source": "mt_urdu"})
del _d1

# ── DS2: LEGAL-UQA  (3 pairs per row: question, answer, context) ──────────────
_d2 = load_dataset("nlp-anonymous-researcher/LEGAL-UQA")
_d2 = concatenate_datasets([_d2[s] for s in _d2.keys()])

def expand_legal(row):
    pairs = []
    for en_k, ur_k in [("question_eng","question_urdu"),
                        ("answer_eng","answer_urdu"),
                        ("context_eng","context_urdu")]:
        en = (row.get(en_k) or "").strip()
        ur = (row.get(ur_k) or "").strip()
        if en and ur:
            pairs.append({"en": en, "ur": ur, "source": "legal_uqa"})
    return pairs

_legal_rows = []
for row in _d2:
    _legal_rows.extend(expand_legal(row))
ds2_full = HFDataset.from_list(_legal_rows)
del _d2, _legal_rows

# ── DS3: SUP-NatInst alignment dataset ───────────────────────────────────────
_ALIGN_RE = re.compile(
    r"Now complete the following example\s*-\s*\nInput:\s*(.+?)\nOutput:",
    re.DOTALL,
)
_d3 = load_from_disk("alignment_dataset")
_d3 = concatenate_datasets([_d3[s] for s in _d3.keys()])

def parse_alignment(row):
    out_list = row.get("output") or []
    ur = out_list[0].strip() if out_list else ""
    m  = _ALIGN_RE.search(row.get("input", ""))
    en = m.group(1).strip() if m else ""
    return {"en": en, "ur": ur, "source": "alignment"}

ds3_full = _d3.map(parse_alignment, remove_columns=_d3.column_names)
ds3_full = ds3_full.filter(lambda r: bool(r["en"]) and bool(r["ur"]))
del _d3

# ── DS4: Urdu idioms ──────────────────────────────────────────────────────────
_d4 = load_dataset("irtizaab/Urdu-English_Transliteration")
_d4 = concatenate_datasets([_d4[s] for s in _d4.keys()])

def parse_idiom(row):
    raw = (row.get("text") or "").strip()
    if ":" not in raw:
        return {"en": "", "ur": "", "source": "idioms"}
    ur, en = raw.split(":", 1)
    return {"en": en.strip(), "ur": ur.strip(), "source": "idioms"}

ds4_full = _d4.map(parse_idiom, remove_columns=_d4.column_names)
ds4_full = ds4_full.filter(lambda r: bool(r["en"]) and bool(r["ur"]))
del _d4

print(f"\n✅ Raw sizes:")
print(f"   DS1 (mt_urdu)  : {len(ds1_full):,}")
print(f"   DS2 (legal_uqa): {len(ds2_full):,}")
print(f"   DS3 (alignment): {len(ds3_full):,}")
print(f"   DS4 (idioms)   : {len(ds4_full):,}")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Clean all datasets & build equal-contribution combined dataset
# ══════════════════════════════════════════════════════════════════════════════
print("Cleaning datasets...")
ds1_clean = clean_dataset(ds1_full)
ds2_clean = clean_dataset(ds2_full)
ds3_clean = clean_dataset(ds3_full)
ds4_clean = clean_dataset(ds4_full)

# Equal contributions: sample N from each where N = min dataset size (capped at 2000)
N_PER_DS  = min(2000, len(ds1_clean), len(ds2_clean), len(ds3_clean), len(ds4_clean))
print(f"\n✅ Equal contribution per dataset: {N_PER_DS:,} records")

def sample_n(ds, n):
    return ds.shuffle(seed=SEED).select(range(n))

ds1_eq = sample_n(ds1_clean, N_PER_DS)
ds2_eq = sample_n(ds2_clean, N_PER_DS)
ds3_eq = sample_n(ds3_clean, N_PER_DS)
ds4_eq = sample_n(ds4_clean, N_PER_DS)

combined = concatenate_datasets([ds1_eq, ds2_eq, ds3_eq, ds4_eq]).shuffle(seed=SEED)
comb_split = combined.train_test_split(test_size=0.15, seed=SEED)
train_combined = comb_split["train"]
test_combined  = comb_split["test"]

print(f"\n✅ Combined dataset (equal contributions):")
print(f"   Total : {len(combined):,}  |  Train: {len(train_combined):,}  |  Test: {len(test_combined):,}")

# Exp 1 & 2: MT-Urdu-English only (unclean and clean), same size as combined train
N_MTONLY = len(train_combined)
train_unclean = ds1_full.shuffle(seed=SEED).select(range(N_MTONLY))
test_unclean  = ds1_full.shuffle(seed=SEED+1).select(range(len(test_combined)))
train_clean   = ds1_clean.shuffle(seed=SEED).select(range(N_MTONLY))
test_clean    = ds1_clean.shuffle(seed=SEED+1).select(range(len(test_combined)))

print(f"\n✅ MT-Urdu-English (Exp 1 & 2):")
print(f"   Train: {len(train_clean):,}  |  Test: {len(test_clean):,}")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Tokeniser & tokenisation helpers
# ══════════════════════════════════════════════════════════════════════════════
nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_ID)
nllb_tokenizer.src_lang = SRC_LANG

def tokenize_nllb(batch):
    model_inputs = nllb_tokenizer(
        batch["en"], max_length=MAX_LEN, truncation=True, padding=False)
    labels = nllb_tokenizer(
        text_target=batch["ur"], max_length=MAX_LEN, truncation=True, padding=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

DROP = ["en", "ur", "source"]

def tokenize_pair(train_ds, test_ds) -> dict:
    tok = {}
    for name, ds in [("train", train_ds), ("test", test_ds)]:
        cols = [c for c in DROP if c in ds.column_names]
        tok[name] = ds.map(tokenize_nllb, batched=True,
                           remove_columns=cols, desc=f"Tokenising {name}")
    return tok

print("✅ Tokeniser ready")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Evaluation metrics: BLEU + ROUGE + METEOR + chrF
# ══════════════════════════════════════════════════════════════════════════════
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

def compute_all_metrics(preds: list[str], refs: list[str]) -> dict:
    """Compute BLEU, ROUGE-L, METEOR, chrF for a list of predictions vs references."""
    bleu  = sacrebleu.corpus_bleu(preds, [[r] for r in refs]).score
    chrf  = sacrebleu.corpus_chrf(preds,  [[r] for r in refs]).score
    meteor_scores = [
        nltk.translate.meteor_score.single_meteor_score(
            nltk.word_tokenize(r), nltk.word_tokenize(p))
        for p, r in zip(preds, refs)
    ]
    meteor = float(np.mean(meteor_scores)) * 100
    rouge_l = float(np.mean([
        _rouge.score(r, p)["rougeL"].fmeasure
        for p, r in zip(preds, refs)
    ])) * 100

    return {
        "BLEU":    round(bleu,    2),
        "chrF":    round(chrf,    2),
        "METEOR":  round(meteor,  2),
        "ROUGE-L": round(rouge_l, 2),
    }

def compute_metrics_for_trainer(eval_preds):
    """Lightweight BLEU-only metric for the HF Trainer (fast)."""
    preds, labels = eval_preds
    labels = [[(t if t != -100 else nllb_tokenizer.pad_token_id) for t in l] for l in labels]
    decoded_preds  = nllb_tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = nllb_tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = sacrebleu.corpus_bleu(decoded_preds, [[r] for r in decoded_labels])
    return {"bleu": round(result.score, 4)}

print("✅ Metrics ready")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Training helper
# ══════════════════════════════════════════════════════════════════════════════
def train_nllb_lora(tokenized: dict, output_dir: str, label: str) -> float:
    """Train NLLB+LoRA on tokenized data. Returns best eval BLEU."""
    print(f"\n{'━'*55}\n  Training: {label}\n{'━'*55}")

    base  = AutoModelForSeq2SeqLM.from_pretrained(NLLB_ID, device_map="auto")
    model = get_peft_model(base, LoraConfig(**LORA_CFG))
    model.print_trainable_parameters()

    collator = DataCollatorForSeq2Seq(nllb_tokenizer, model=model, padding=True)
    args     = Seq2SeqTrainingArguments(output_dir=output_dir, **TRAIN_ARGS)

    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=tokenized["train"], eval_dataset=tokenized["test"],
        processing_class=nllb_tokenizer, data_collator=collator,
        compute_metrics=compute_metrics_for_trainer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()
    trainer.save_model(output_dir)

    metrics = trainer.evaluate(tokenized["test"])
    bleu    = metrics.get("eval_bleu", 0.0)
    print(f"✅ {label} — Trainer BLEU: {bleu:.2f}")

    del model, trainer, base
    gc.collect(); torch.cuda.empty_cache()
    return bleu

print("✅ train_nllb_lora() ready")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 10 — Inference helper (NLLB LoRA)
# ══════════════════════════════════════════════════════════════════════════════
def load_nllb_for_inference(output_dir: str):
    base  = AutoModelForSeq2SeqLM.from_pretrained(NLLB_ID, device_map="auto")
    model = PeftModel.from_pretrained(base, output_dir)
    model = model.merge_and_unload()
    model.eval()
    return model

def translate_batch(model, sentences: list[str], tokenizer=None, xlmr=False) -> list[str]:
    """Translate a list of EN sentences to UR."""
    tok     = tokenizer or nllb_tokenizer
    device  = next(model.parameters()).device
    preds   = []
    forced_bos = tok.convert_tokens_to_ids(TGT_LANG) if not xlmr else None

    for sent in sentences:
        inputs = tok(sent, return_tensors="pt",
                     max_length=MAX_LEN, truncation=True).to(device)
        with torch.no_grad():
            if xlmr:
                out = model.generate(**inputs, max_new_tokens=MAX_LEN, num_beams=4)
            else:
                out = model.generate(**inputs,
                                     forced_bos_token_id=forced_bos,
                                     max_new_tokens=MAX_LEN, num_beams=4)
        preds.append(tok.decode(out[0], skip_special_tokens=True))
    return preds

print("✅ Inference helpers ready")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 11 — Full evaluation (metrics + samples + error analysis)
# ══════════════════════════════════════════════════════════════════════════════
def full_eval(model, test_ds: HFDataset, label: str,
              tokenizer=None, xlmr=False) -> dict:
    """Run all metrics + error analysis on test_ds. Returns metrics dict."""
    print(f"\n{'━'*55}\n  Evaluating: {label}\n{'━'*55}")

    sample = test_ds.shuffle(seed=SEED).select(range(min(N_ERROR, len(test_ds))))
    preds  = translate_batch(model, list(sample["en"]), tokenizer, xlmr)
    refs   = list(sample["ur"])

    metrics = compute_all_metrics(preds, refs)
    print(f"\n  BLEU   : {metrics['BLEU']:.2f}")
    print(f"  chrF   : {metrics['chrF']:.2f}")
    print(f"  METEOR : {metrics['METEOR']:.2f}")
    print(f"  ROUGE-L: {metrics['ROUGE-L']:.2f}")

    # ── Sample outputs ────────────────────────────────────────────────────────
    print(f"\n  {'─'*50}")
    print(f"  SAMPLE TRANSLATIONS")
    print(f"  {'─'*50}")
    idxs = random.sample(range(len(sample)), min(N_SAMPLES, len(sample)))
    for i, idx in enumerate(idxs, 1):
        bleu_s = sacrebleu.sentence_bleu(preds[idx], [refs[idx]]).score
        print(f"\n  [{i}] EN  : {sample[idx]['en']}")
        print(f"       REF : {refs[idx]}")
        print(f"       PRED: {preds[idx]}")
        print(f"       BLEU: {bleu_s:.2f}")

    # ── Error analysis by sentence length ─────────────────────────────────────
    print(f"\n  {'─'*50}")
    print(f"  ERROR ANALYSIS — by sentence length")
    print(f"  {'─'*50}")
    buckets = {"short (1-5)": [], "medium (6-20)": [], "long (21+)": []}
    for en, pred, ref in zip(sample["en"], preds, refs):
        n = len(en.split())
        k = "short (1-5)" if n <= 5 else ("medium (6-20)" if n <= 20 else "long (21+)")
        buckets[k].append((en, pred, ref))

    bucket_bleus = {}
    for bname, rows in buckets.items():
        if not rows: continue
        bp, br = [r[1] for r in rows], [r[2] for r in rows]
        b = sacrebleu.corpus_bleu(bp, [[r] for r in br]).score
        bucket_bleus[bname] = round(b, 2)
        print(f"\n  [{bname}]  BLEU = {b:.2f}  (n={len(rows)})")
        worst = min(rows, key=lambda x: sacrebleu.sentence_bleu(x[1],[x[2]]).score)
        ws = sacrebleu.sentence_bleu(worst[1], [worst[2]]).score
        print(f"    Worst (BLEU={ws:.2f}): {worst[0]}")
        print(f"    REF : {worst[2]}")
        print(f"    PRED: {worst[1]}")

    metrics["bucket_bleus"] = bucket_bleus
    return metrics

print("✅ full_eval() ready")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 12 — XLM-R + NLLB decoder architecture
# ══════════════════════════════════════════════════════════════════════════════
XLMR_DIM = 768    # xlm-roberta-base hidden size
NLLB_DIM = 1024   # nllb-200-distilled-600M decoder hidden size

class XLMRNLLBModel(nn.Module):
    """
    Custom encoder-decoder:
      Encoder : XLM-RoBERTa (frozen)
      Bridge  : Linear projection 768 → 1024
      Decoder : NLLB decoder with LoRA (trainable)
    """
    def __init__(self, xlmr, nllb):
        super().__init__()
        self.xlmr_encoder  = xlmr.roberta          # XLM-R body
        self.projection    = nn.Linear(XLMR_DIM, NLLB_DIM)
        self.nllb_decoder  = nllb.model.decoder
        self.lm_head       = nllb.lm_head
        self.nllb_config   = nllb.config
        self.vocab_size    = nllb.config.vocab_size

        # Freeze XLM-R — only projection + decoder will train
        for p in self.xlmr_encoder.parameters():
            p.requires_grad = False

    def get_encoder_output(self, input_ids, attention_mask):
        enc = self.xlmr_encoder(input_ids=input_ids,
                                attention_mask=attention_mask)
        return self.projection(enc.last_hidden_state)

    def forward(self, input_ids, attention_mask,
                decoder_input_ids, labels=None):
        enc_hidden = self.get_encoder_output(input_ids, attention_mask)

        dec_out = self.nllb_decoder(
            input_ids             = decoder_input_ids,
            encoder_hidden_states = enc_hidden,
            encoder_attention_mask= attention_mask,
        )
        logits = self.lm_head(dec_out.last_hidden_state)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss(ignore_index=-100)(
                logits.view(-1, self.vocab_size), labels.view(-1))
        return loss, logits

    @torch.no_grad()
    def generate(self, input_ids, attention_mask,
                 forced_bos_token_id, max_new_tokens=128, num_beams=4):
        """Greedy / beam decode for inference."""
        device     = input_ids.device
        enc_hidden = self.get_encoder_output(input_ids, attention_mask)
        bos_id     = self.nllb_config.decoder_start_token_id

        # Start with BOS
        dec_ids = torch.tensor([[bos_id, forced_bos_token_id]],
                               device=device)
        eos_id  = self.nllb_config.eos_token_id

        for _ in range(max_new_tokens):
            dec_out = self.nllb_decoder(
                input_ids             = dec_ids,
                encoder_hidden_states = enc_hidden,
                encoder_attention_mask= attention_mask,
            )
            next_id = self.lm_head(dec_out.last_hidden_state[:, -1:]).argmax(-1)
            dec_ids = torch.cat([dec_ids, next_id], dim=-1)
            if next_id.item() == eos_id:
                break
        return dec_ids

print("✅ XLMRNLLBModel defined")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 13 — XLM-R tokeniser & tokenisation for Exp 4
# ══════════════════════════════════════════════════════════════════════════════
xlmr_tokenizer = AutoTokenizer.from_pretrained(XLMR_ID)

def tokenize_xlmr(batch):
    """Tokenise EN with XLM-R, UR (labels) with NLLB tokeniser."""
    enc = xlmr_tokenizer(
        batch["en"], max_length=MAX_LEN, truncation=True, padding=False)
    with nllb_tokenizer.as_target_tokenizer() if hasattr(
            nllb_tokenizer, "as_target_tokenizer") else open(os.devnull):
        lbl = nllb_tokenizer(
            text_target=batch["ur"], max_length=MAX_LEN,
            truncation=True, padding=False)
    enc["labels"] = lbl["input_ids"]
    return enc

def tokenize_xlmr_pair(train_ds, test_ds) -> dict:
    tok = {}
    for name, ds in [("train", train_ds), ("test", test_ds)]:
        cols = [c for c in DROP if c in ds.column_names]
        tok[name] = ds.map(tokenize_xlmr, batched=True,
                           remove_columns=cols, desc=f"Tokenising {name} (XLM-R)")
    return tok

print("✅ XLM-R tokeniser ready")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 14 — XLM-R + NLLB training loop
# ══════════════════════════════════════════════════════════════════════════════
def train_xlmr_nllb(train_ds, test_ds, output_dir, label) -> float:
    """Train XLM-R encoder + NLLB LoRA decoder. Returns eval BLEU."""
    print(f"\n{'━'*55}\n  Training: {label}\n{'━'*55}")

    # Load base models
    xlmr_base = AutoModel.from_pretrained(XLMR_ID)
    nllb_base = AutoModelForSeq2SeqLM.from_pretrained(NLLB_ID)

    # Apply LoRA to NLLB decoder attention only
    nllb_lora = get_peft_model(nllb_base, LoraConfig(**LORA_CFG))

    model = XLMRNLLBModel(xlmr_base, nllb_lora).to("cuda")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

    # Tokenise
    tokenized = tokenize_xlmr_pair(train_ds, test_ds)

    # Custom training loop (HF Trainer doesn't support custom forward easily)
    from torch.utils.data import DataLoader
    from transformers import get_linear_schedule_with_warmup

    collator = DataCollatorForSeq2Seq(nllb_tokenizer, model=model, padding=True)

    def collate(batch):
        return {k: torch.tensor([b[k] for b in batch]).to("cuda")
                for k in batch[0] if k in ("input_ids","attention_mask","labels")}

    train_loader = DataLoader(tokenized["train"], batch_size=4,
                              shuffle=True, collate_fn=collate)
    optimizer    = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=3e-4)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=100,
        num_training_steps=len(train_loader) * TRAIN_ARGS["num_train_epochs"])

    model.train()
    total_loss = 0
    steps = 0
    for batch in train_loader:
        inp   = batch["input_ids"]
        mask  = batch["attention_mask"]
        lbl   = batch["labels"]

        # Shift labels right for decoder input
        dec_inp = torch.full_like(lbl, nllb_tokenizer.pad_token_id)
        dec_inp[:, 1:] = lbl[:, :-1].clone()
        dec_inp[:, 0]  = nllb_base.config.decoder_start_token_id

        loss, _ = model(inp, mask, dec_inp, lbl)
        loss.backward()

        if (steps + 1) % 4 == 0:
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        total_loss += loss.item()
        steps += 1
        if steps % 100 == 0:
            print(f"  Step {steps}  loss={total_loss/steps:.4f}")

    # Save
    Path(output_dir).mkdir(exist_ok=True)
    torch.save(model.state_dict(), f"{output_dir}/model.pt")
    print(f"✅ Saved → {output_dir}/model.pt")

    # Quick BLEU eval
    model.eval()
    forced_bos = nllb_tokenizer.convert_tokens_to_ids(TGT_LANG)
    test_sample = test_ds.shuffle(seed=SEED).select(range(min(200, len(test_ds))))
    preds, refs = [], []
    for row in test_sample:
        inp = xlmr_tokenizer(row["en"], return_tensors="pt",
                             max_length=MAX_LEN, truncation=True).to("cuda")
        out = model.generate(inp["input_ids"], inp["attention_mask"],
                             forced_bos_token_id=forced_bos)
        preds.append(nllb_tokenizer.decode(out[0], skip_special_tokens=True))
        refs.append(row["ur"])

    bleu = sacrebleu.corpus_bleu(preds, [[r] for r in refs]).score
    print(f"✅ {label} — BLEU: {bleu:.2f}")

    del xlmr_base, nllb_base, nllb_lora
    gc.collect(); torch.cuda.empty_cache()
    return model, round(bleu, 2)

print("✅ train_xlmr_nllb() ready")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 15 — RUN EXPERIMENT 1: Baseline unclean (MT-Urdu only)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*55)
print("  EXPERIMENT 1 — Baseline (unclean, MT-Urdu only)")
print("═"*55)

tok_exp1   = tokenize_pair(train_unclean, test_unclean)
train_nllb_lora(tok_exp1, DIR_EXP1, "Exp1 Unclean")
model_exp1 = load_nllb_for_inference(DIR_EXP1)
results_exp1 = full_eval(model_exp1, test_unclean, "Exp1 — Unclean Baseline")
del model_exp1; gc.collect(); torch.cuda.empty_cache()


# ══════════════════════════════════════════════════════════════════════════════
# CELL 16 — RUN EXPERIMENT 2: Baseline clean (MT-Urdu only)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*55)
print("  EXPERIMENT 2 — Baseline (clean, MT-Urdu only)")
print("═"*55)

tok_exp2   = tokenize_pair(train_clean, test_clean)
train_nllb_lora(tok_exp2, DIR_EXP2, "Exp2 Clean")
model_exp2 = load_nllb_for_inference(DIR_EXP2)
results_exp2 = full_eval(model_exp2, test_clean, "Exp2 — Clean Baseline")
del model_exp2; gc.collect(); torch.cuda.empty_cache()


# ══════════════════════════════════════════════════════════════════════════════
# CELL 17 — RUN EXPERIMENT 3: Combined 4 datasets (equal contributions)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*55)
print("  EXPERIMENT 3 — Combined datasets (equal contributions)")
print("═"*55)

tok_exp3   = tokenize_pair(train_combined, test_combined)
train_nllb_lora(tok_exp3, DIR_EXP3, "Exp3 Combined")
model_exp3 = load_nllb_for_inference(DIR_EXP3)
results_exp3 = full_eval(model_exp3, test_combined, "Exp3 — Combined Datasets")
del model_exp3; gc.collect(); torch.cuda.empty_cache()


# ══════════════════════════════════════════════════════════════════════════════
# CELL 18 — RUN EXPERIMENT 4: XLM-R encoder + NLLB decoder
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*55)
print("  EXPERIMENT 4 — XLM-R encoder + NLLB decoder")
print("═"*55)

model_exp4, bleu_exp4 = train_xlmr_nllb(
    train_combined, test_combined, DIR_EXP4, "Exp4 XLM-R+NLLB")

# Full eval — reuse translate_batch with XLM-R tokeniser
forced_bos_exp4 = nllb_tokenizer.convert_tokens_to_ids(TGT_LANG)
test_sample4    = test_combined.shuffle(seed=SEED).select(range(min(N_ERROR, len(test_combined))))
preds_exp4      = []
device4         = next(model_exp4.parameters()).device
for row in test_sample4:
    inp = xlmr_tokenizer(row["en"], return_tensors="pt",
                         max_length=MAX_LEN, truncation=True).to(device4)
    out = model_exp4.generate(inp["input_ids"], inp["attention_mask"],
                              forced_bos_token_id=forced_bos_exp4)
    preds_exp4.append(nllb_tokenizer.decode(out[0], skip_special_tokens=True))

refs_exp4    = list(test_sample4["ur"])
results_exp4 = compute_all_metrics(preds_exp4, refs_exp4)
results_exp4["bucket_bleus"] = {}
print(f"\n  BLEU   : {results_exp4['BLEU']:.2f}")
print(f"  chrF   : {results_exp4['chrF']:.2f}")
print(f"  METEOR : {results_exp4['METEOR']:.2f}")
print(f"  ROUGE-L: {results_exp4['ROUGE-L']:.2f}")
del model_exp4; gc.collect(); torch.cuda.empty_cache()


# ══════════════════════════════════════════════════════════════════════════════
# CELL 19 — Summary table & comparison plots
# ══════════════════════════════════════════════════════════════════════════════
experiments = {
    "Exp1\nUnclean":  results_exp1,
    "Exp2\nClean":    results_exp2,
    "Exp3\nCombined": results_exp3,
    "Exp4\nXLM-R":    results_exp4,
}
metrics_list = ["BLEU", "chrF", "METEOR", "ROUGE-L"]

# ── Print summary table ───────────────────────────────────────────────────────
print(f"\n{'━'*65}")
print(f"  FINAL RESULTS SUMMARY")
print(f"{'━'*65}")
header = f"{'Experiment':<20}" + "".join(f"{m:>10}" for m in metrics_list)
print(header)
print("─" * 65)
for exp_name, res in experiments.items():
    row = f"{exp_name.replace(chr(10),' '):<20}"
    row += "".join(f"{res[m]:>10.2f}" for m in metrics_list)
    print(row)
print(f"{'━'*65}")

# ── Plot 1: All metrics bar chart ─────────────────────────────────────────────
exp_labels = [e.replace("\n", "\n") for e in experiments.keys()]
colors     = ["#DD8452", "#4C72B0", "#55A868", "#C44E52"]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Experiment Comparison — EN→UR NMT\n(NLLB-600M LoRA)", fontsize=14, fontweight="bold")

for ax, metric in zip(axes.flat, metrics_list):
    vals = [experiments[e][metric] for e in experiments]
    bars = ax.bar(exp_labels, vals, color=colors, width=0.5)
    ax.set_title(metric, fontweight="bold")
    ax.set_ylabel(metric)
    ax.set_ylim(0, max(vals) * 1.3 if max(vals) > 0 else 1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{val:.2f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig("all_metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → all_metrics_comparison.png")

# ── Plot 2: BLEU by sentence length ──────────────────────────────────────────
bucket_names = ["short (1-5)", "medium (6-20)", "long (21+)"]
exp_bucket_data = {
    "Exp1 Unclean":  results_exp1.get("bucket_bleus", {}),
    "Exp2 Clean":    results_exp2.get("bucket_bleus", {}),
    "Exp3 Combined": results_exp3.get("bucket_bleus", {}),
}

fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(bucket_names))
width = 0.25
for i, (exp_label, bbleus) in enumerate(exp_bucket_data.items()):
    vals = [bbleus.get(b, 0) for b in bucket_names]
    bars = ax.bar(x + i*width, vals, width, label=exp_label, color=colors[i])
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                    f"{val:.1f}", ha="center", fontsize=8)

ax.set_title("BLEU by Sentence Length — Exp 1, 2, 3", fontweight="bold")
ax.set_ylabel("Corpus BLEU")
ax.set_xticks(x + width)
ax.set_xticklabels(bucket_names)
ax.legend()
plt.tight_layout()
plt.savefig("bleu_by_length.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → bleu_by_length.png")

# ── Save all results to JSON ──────────────────────────────────────────────────
final_results = {
    k.replace("\n", "_"): v for k, v in experiments.items()
}
with open("final_results.json", "w") as f:
    json.dump(final_results, f, indent=2)
print("✅ Saved → final_results.json")